# 6 - Máscaras — protocolo de mascaramento da GAIN

## Objetivo

Estabelecer o protocolo central de mascaramento usado pela GAIN:

1. **Máscara real `M`** — uma por split, `1 = observado / 0 = faltante`. Reflete a esparsidade do dataset.
2. **Função geradora da máscara artificial `B`** — chamada a cada batch durante o treino para forçar o gerador a reconstruir células que sabemos estarem corretas. Em runtime, o gerador é treinado a recuperar `(M=1, B=0)` (artificialmente mascaradas) e o discriminador a distinguir essas das `(M=1, B=1)` (mantidas).

## Posição na Etapa 2

Notebook **6 de 6** — fim do pré-processamento. Consome os três splits + `encoded_columns.json`. Entrega `mask_real_{train,val,test}.parquet` + `mask_utils.py` (módulo importável).

## Decisões registradas

### Imputar apenas as 13 variáveis numéricas

`M` e `B` operam **apenas** sobre as colunas das 13 variáveis numéricas. Não fazem sentido para:

- **One-hot de estação** (`est_*`): o modelo não deve "inventar" uma estação de coleta.
- **One-hot de censura** (`<Var>_LD_*`): a informação de censura é estrutural, não imputável — quem tem `<` continua com `<`.
- **Features temporais** (`ano_norm`, `Mes_sin/cos`, `umido`, `dias_desde_inicio`): por construção, nunca são `NaN`.
- **Identificadores** (`Data`, `Ano_int`): metadados, fora do vetor da GAIN.

### Taxa de máscara artificial **por variável**, não global

A cobertura real do dataset é fortemente desigual (ver `Code/2 - EDA/resumo.md` §1):

| Categoria | Variáveis | `miss_rate` sugerido |
|---|---|---:|
| Robustas (cobertura ≥ 95%) | DBO, OD, pH, Turbidez, Temperatura, Condutividade, Fósforo Total | 0,20 |
| Intermediárias (50–90%) | Nitrato, Nitrogênio Amoniacal Total, Coliformes Termotolerantes | 0,10 |
| Críticas (< 25%) | Sólidos Suspensos Totais, Cianobactérias, Microcistinas | 0,05 |

Aplicar `0,2` uniforme em variáveis críticas (Microcistinas tem só 52 observações) apagaria uma fração desproporcional do pouco sinal disponível — atrapalhando o discriminador. A calibração inicial usa taxas escalonadas; pode ser refinada em `04_GAIN/02_treinamento.ipynb` (early stopping por val loss).

### `B` é gerado a cada batch, não persistido

Para train/val: `generate_artificial_mask(M, miss_rate, seed=None)` chamada em runtime — diferente em cada batch, dá robustez ao gerador. Para test (avaliação): chamada com `seed` fixa para reprodutibilidade entre GAIN × baselines.


## Setup

Imports + caminhos relativos a `Code/3 - Preprocessing/06_mascaras.ipynb`.

In [1]:
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

IN_SPLIT_DIR    = Path("../../Data/GoldData/Splited")
IN_COLUMNS_JSON = Path("../../Data/ProcessedData/encoded_columns.json")

OUT_MASK_DIR    = Path("../../Data/GoldData/Masked")
OUT_MASK_DIR.mkdir(parents=True, exist_ok=True)

# Módulo importável fica em Code/ (convenção pythonica)
OUT_MASK_UTILS  = Path("./mask_utils.py")

## Colunas-alvo da imputação

As 13 variáveis numéricas de `encoded_columns.json["numericas"]`. Verificamos também que `umido`, `Mes_sin/cos`, `ano_norm` e `dias_desde_inicio` realmente não têm NaN (precondição do design).

In [2]:
with open(IN_COLUMNS_JSON, "r", encoding="utf-8") as f:
    encoded_columns = json.load(f)

VARS = encoded_columns["numericas"]
print(f"Variáveis-alvo da imputação ({len(VARS)}):")
for v in VARS:
    print(f"  - {v}")

# Carregar os splits
df_train = pd.read_parquet(IN_SPLIT_DIR / "train.parquet")
df_val   = pd.read_parquet(IN_SPLIT_DIR / "val.parquet")
df_test  = pd.read_parquet(IN_SPLIT_DIR / "test.parquet")

# Sanidade: features temporais e binárias não devem ter NaN
cols_nao_imputaveis = encoded_columns["temporais"] + encoded_columns["estacao_onehot"] + encoded_columns["ld_onehot"]
for nome, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    nan_indevido = d[cols_nao_imputaveis].isna().sum().sum()
    assert nan_indevido == 0, f"NaN em colunas não-alvo de {nome}: {nan_indevido}"
print("\nNenhum NaN em colunas não-alvo nos três splits — precondição OK.")

Variáveis-alvo da imputação (13):
  - DBO
  - OD
  - Nitrato
  - Nitrogênio Amoniacal Total
  - Fósforo Total
  - Condutividade
  - pH
  - Turbidez
  - Temperatura da Água
  - Sólidos Suspensos Totais
  - Coliformes Termotolerantes
  - Cianobacterias
  - Microcistinas

Nenhum NaN em colunas não-alvo nos três splits — precondição OK.


## Máscara real `M`

`M[i, j] = 1` se a célula `(i, j)` é observada (não-NaN), `0` se faltante. Persistimos como `int8` (1 byte/célula) — booleano em parquet via PyArrow tem ganho marginal e adiciona acoplamento ao backend.

A taxa de células observadas (`M.mean()`) é o **complemento** da esparsidade do dataset bruto — é o orçamento real do que a GAIN tem para aprender.

In [3]:
def construir_mask_real(df, target_cols):
    M = (~df[target_cols].isna()).astype("int8")
    return M

mask_real_train = construir_mask_real(df_train, VARS)
mask_real_val   = construir_mask_real(df_val,   VARS)
mask_real_test  = construir_mask_real(df_test,  VARS)

# Estatísticas globais
for nome, M in [("train", mask_real_train), ("val", mask_real_val), ("test", mask_real_test)]:
    total = M.size
    obs   = int(M.sum().sum())
    print(f"{nome:5s}: {obs}/{total} células observadas ({100*obs/total:.1f}%) — shape {M.shape}")

train: 5103/6929 células observadas (73.6%) — shape (533, 13)
val  : 327/442 células observadas (74.0%) — shape (34, 13)
test : 675/1170 células observadas (57.7%) — shape (90, 13)


### Cobertura por variável e split

A heterogeneidade de cobertura entre variáveis é exatamente o que motiva o `miss_rate` por variável. Variáveis com cobertura muito baixa em algum split (ex.: Cianobactérias ausente em val/test) já foram registradas como gaps conscientes em `05_split.ipynb`.

In [4]:
cobertura = pd.DataFrame({
    "train": mask_real_train.sum() / len(mask_real_train),
    "val":   mask_real_val.sum()   / len(mask_real_val)   if len(mask_real_val)   > 0 else np.nan,
    "test":  mask_real_test.sum()  / len(mask_real_test)  if len(mask_real_test)  > 0 else np.nan,
})
cobertura.index.name = "variavel"
cobertura.round(3)

,train,val,test
variavel,,,
DBO,0.9940,1.0000,0.8670
OD,0.9890,1.0000,0.8220
Nitrato,0.4730,0.3530,0.8110
Nitrogênio Amoniacal Total,0.7240,1.0000,0.8560
Fósforo Total,0.9940,1.0000,0.6670
Condutividade,0.9960,0.9410,0.8000
pH,0.9870,1.0000,0.8780
Turbidez,0.9890,1.0000,0.7780
Temperatura da Água,0.9740,1.0000,0.8440


## Classificação de variáveis por cobertura — `miss_rate` por categoria

A classificação segue `Code/2 - EDA/resumo.md` §1 e a cobertura **global** observada (não a por-split, que oscila com gaps temporais conhecidos).

In [5]:
# Categorias e taxas por variável (calibração inicial; refinar em 04_GAIN/02_treinamento.ipynb)
ROBUSTAS = ["DBO", "OD", "pH", "Turbidez", "Temperatura da Água",
            "Condutividade", "Fósforo Total"]
INTERMEDIARIAS = ["Nitrato", "Nitrogênio Amoniacal Total", "Coliformes Termotolerantes"]
CRITICAS = ["Sólidos Suspensos Totais", "Cianobacterias", "Microcistinas"]

# Sanidade: união cobre exatamente as 13 variáveis
union = set(ROBUSTAS) | set(INTERMEDIARIAS) | set(CRITICAS)
assert union == set(VARS), f"Categorias não cobrem todas as variáveis: {set(VARS) ^ union}"

MISS_RATES_PADRAO = {
    **{v: 0.20 for v in ROBUSTAS},
    **{v: 0.10 for v in INTERMEDIARIAS},
    **{v: 0.05 for v in CRITICAS},
}
pd.DataFrame({
    "variavel": VARS,
    "categoria": ["robusta" if v in ROBUSTAS else "intermediária" if v in INTERMEDIARIAS else "crítica" for v in VARS],
    "miss_rate": [MISS_RATES_PADRAO[v] for v in VARS],
}).set_index("variavel")

,categoria,miss_rate
variavel,,
DBO,robusta,0.2000
OD,robusta,0.2000
Nitrato,intermediária,0.1000
Nitrogênio Amoniacal Total,intermediária,0.1000
Fósforo Total,robusta,0.2000
Condutividade,robusta,0.2000
pH,robusta,0.2000
Turbidez,robusta,0.2000
Temperatura da Água,robusta,0.2000


## Função `generate_artificial_mask`

Definida **inline** no notebook (para validar comportamento) e **persistida** em `Code/3 - Preprocessing/mask_utils.py` como módulo importável.

### Contrato

```
B = generate_artificial_mask(M, miss_rate, seed=None)

M:          array/DataFrame (N, P) com 0/1.
miss_rate:  float (taxa única em todas as colunas) OU
            dict {coluna: taxa} (taxa por variável).
seed:       int ou None. Se int, resultado reprodutível.

Retorno:
B:          mesma forma de M.
            - Onde M=0 → B=0 (nada a esconder; já é NaN).
            - Onde M=1 → B ~ Bernoulli(1 - miss_rate).
                         B=1 mantém, B=0 mascara artificialmente.
```

### Por que `B=0` onde `M=0`?

A definição de "mascarado para a GAIN" é `M ∧ B`. Faz sentido apenas onde temos a verdade (`M=1`). Onde `M=0`, B é irrelevante — fixamos em 0 por convenção para simplificar contagens nas estatísticas (`(M=0, B=0)` = células originalmente faltantes).

In [6]:
def generate_artificial_mask(M, miss_rate, seed=None):
    """Gera B (máscara artificial) com a mesma forma de M.

    M: array (N, P) com 0/1. Tipicamente DataFrame com colunas = VARS.
    miss_rate: float ou dict {coluna: taxa}.
    seed: int para reprodutibilidade.

    Convenção:
    - Onde M==0 → B=0 (já faltante, nada a esconder).
    - Onde M==1 → B ~ Bernoulli(1 - miss_rate); B=1 mantém, B=0 esconde.
    """
    rng = np.random.default_rng(seed)

    if isinstance(M, pd.DataFrame):
        M_arr = M.to_numpy()
        colunas = list(M.columns)
        indice  = M.index
        retorno_df = True
    else:
        M_arr = np.asarray(M)
        colunas = None
        indice  = None
        retorno_df = False

    if isinstance(miss_rate, dict):
        if colunas is None:
            raise ValueError("miss_rate dict requer DataFrame em M para identificar colunas.")
        rates = np.array([miss_rate[c] for c in colunas], dtype=float)
        # taxa por coluna; broadcasting (N, P) <- (P,)
        u = rng.random(M_arr.shape)
        B = (u >= rates[np.newaxis, :]).astype(np.int8)
    else:
        rate = float(miss_rate)
        u = rng.random(M_arr.shape)
        B = (u >= rate).astype(np.int8)

    # Onde M==0, B=0 por convenção
    B = B * (M_arr == 1).astype(np.int8)

    if retorno_df:
        return pd.DataFrame(B, columns=colunas, index=indice)
    return B

### Persistir `mask_utils.py`

Escrevemos o conteúdo do módulo via `textwrap.dedent` num literal — assim a fonte do notebook e o `.py` ficam sincronizados (mesma definição em ambos os lugares ao executar este cell).

In [7]:
MASK_UTILS_SRC = textwrap.dedent('''\
    """Utilidades de mascaramento da GAIN.

    Gerado por Code/3 - Preprocessing/06_mascaras.ipynb. Edite o notebook, não este arquivo.
    """
    from __future__ import annotations

    import numpy as np
    import pandas as pd


    def generate_artificial_mask(M, miss_rate, seed=None):
        """Gera B (máscara artificial) com a mesma forma de M.

        M: array (N, P) com 0/1 ou DataFrame.
        miss_rate: float (taxa única) ou dict {coluna: taxa}.
        seed: int para reprodutibilidade.

        Convenção:
        - Onde M==0 -> B=0 (já faltante, nada a esconder).
        - Onde M==1 -> B ~ Bernoulli(1 - miss_rate); B=1 mantém, B=0 esconde.
        """
        rng = np.random.default_rng(seed)

        if isinstance(M, pd.DataFrame):
            M_arr = M.to_numpy()
            colunas = list(M.columns)
            indice = M.index
            retorno_df = True
        else:
            M_arr = np.asarray(M)
            colunas = None
            indice = None
            retorno_df = False

        if isinstance(miss_rate, dict):
            if colunas is None:
                raise ValueError("miss_rate dict requer DataFrame em M para identificar colunas.")
            rates = np.array([miss_rate[c] for c in colunas], dtype=float)
            u = rng.random(M_arr.shape)
            B = (u >= rates[np.newaxis, :]).astype(np.int8)
        else:
            rate = float(miss_rate)
            u = rng.random(M_arr.shape)
            B = (u >= rate).astype(np.int8)

        B = B * (M_arr == 1).astype(np.int8)

        if retorno_df:
            return pd.DataFrame(B, columns=colunas, index=indice)
        return B
''')

OUT_MASK_UTILS.write_text(MASK_UTILS_SRC, encoding="utf-8")
print(f"Salvo: {OUT_MASK_UTILS.resolve()}")

# Smoke-test do módulo persistido
import importlib, sys
sys.path.insert(0, str(OUT_MASK_UTILS.parent.resolve()))
if "mask_utils" in sys.modules:
    importlib.reload(sys.modules["mask_utils"])
import mask_utils
print(f"Funções expostas: {[n for n in dir(mask_utils) if not n.startswith('_')]}")

Salvo: C:\Users\jhter\OneDrive\Documents\QualiAgua\Code\3 - Preprocessing\mask_utils.py
Funções expostas: ['annotations', 'generate_artificial_mask', 'np', 'pd']


## Demonstração

Aplicar `generate_artificial_mask` sobre `mask_real_train` com:

1. **Taxa única `0,20`** — calibração de referência.
2. **Taxas por variável** — `MISS_RATES_PADRAO`.

Em cada caso, contamos as 3 famílias de células:

- `(M=1, B=1)` — observadas e mantidas (sinal "verdade" para o discriminador).
- `(M=1, B=0)` — observadas e **artificialmente mascaradas** (alvo de reconstrução do gerador).
- `(M=0)` — originalmente faltantes (B=0 por convenção; não entram na loss de reconstrução).

In [8]:
def estatisticas_mascaramento(M, B):
    M_arr = M.to_numpy() if isinstance(M, pd.DataFrame) else np.asarray(M)
    B_arr = B.to_numpy() if isinstance(B, pd.DataFrame) else np.asarray(B)
    total = M_arr.size
    obs_mantidas    = int(((M_arr == 1) & (B_arr == 1)).sum())
    obs_mascaradas  = int(((M_arr == 1) & (B_arr == 0)).sum())
    falt_originais  = int((M_arr == 0).sum())
    return {
        "total_celulas":    total,
        "(M=1, B=1) mantidas":     obs_mantidas,
        "(M=1, B=0) mascaradas":   obs_mascaradas,
        "(M=0) faltantes orig":    falt_originais,
        "%_mascaradas_sobre_obs":  100 * obs_mascaradas / max(1, obs_mantidas + obs_mascaradas),
    }

# 1) Taxa única
B_uniforme = generate_artificial_mask(mask_real_train, miss_rate=0.20, seed=42)
est_uniforme = estatisticas_mascaramento(mask_real_train, B_uniforme)

# 2) Taxas por variável
B_por_var = generate_artificial_mask(mask_real_train, miss_rate=MISS_RATES_PADRAO, seed=42)
est_por_var = estatisticas_mascaramento(mask_real_train, B_por_var)

pd.DataFrame({"uniforme_0.20": est_uniforme, "por_variavel": est_por_var})

,uniforme_0.20,por_variavel
total_celulas,6929.0000,6929.0000
"(M=1, B=1) mantidas",4035.0000,4219.0000
"(M=1, B=0) mascaradas",1068.0000,884.0000
(M=0) faltantes orig,1826.0000,1826.0000
%_mascaradas_sobre_obs,20.9289,17.3231


### Reprodutibilidade

Duas chamadas com a mesma `seed` devem retornar exatamente o mesmo `B`. Duas chamadas com `seed` diferentes devem diferir.

In [9]:
B1 = generate_artificial_mask(mask_real_train, miss_rate=0.20, seed=42)
B2 = generate_artificial_mask(mask_real_train, miss_rate=0.20, seed=42)
B3 = generate_artificial_mask(mask_real_train, miss_rate=0.20, seed=43)

assert (B1.values == B2.values).all(), "seed=42 deveria reproduzir o mesmo B."
assert not (B1.values == B3.values).all(), "seeds diferentes deveriam produzir B diferentes."
print("OK — reprodutibilidade por seed e variação entre seeds confirmadas.")

OK — reprodutibilidade por seed e variação entre seeds confirmadas.


### Taxa efetiva por variável (sanity)

Conferindo que, na média, a fração mascarada artificialmente de cada coluna se aproxima do `miss_rate` configurado. Variações pequenas são esperadas (amostragem Bernoulli sobre pouco dado em variáveis críticas).

In [10]:
obs_por_var       = mask_real_train.sum()
mask_artif_por_var = ((mask_real_train == 1) & (B_por_var == 0)).sum()
taxa_efetiva = (mask_artif_por_var / obs_por_var.replace(0, np.nan)).rename("taxa_efetiva")
taxa_alvo    = pd.Series(MISS_RATES_PADRAO, name="taxa_alvo")
pd.concat([taxa_alvo, taxa_efetiva], axis=1).round(4)

,taxa_alvo,taxa_efetiva
DBO,0.2000,0.2000
OD,0.2000,0.2258
pH,0.2000,0.2167
Turbidez,0.2000,0.2049
Temperatura da Água,0.2000,0.1927
Condutividade,0.2000,0.2090
Fósforo Total,0.2000,0.2208
Nitrato,0.1000,0.0794
Nitrogênio Amoniacal Total,0.1000,0.0829
Coliformes Termotolerantes,0.1000,0.0813


## Persistência das máscaras reais

Três parquets em `Data/GoldData/Masked/`. As máscaras têm **a mesma ordem de linhas** dos splits correspondentes — `mask_real_train.iloc[i]` corresponde a `train.iloc[i]`.

In [11]:
mask_real_train.to_parquet(OUT_MASK_DIR / "mask_real_train.parquet", index=False)
mask_real_val.to_parquet(  OUT_MASK_DIR / "mask_real_val.parquet",   index=False)
mask_real_test.to_parquet( OUT_MASK_DIR / "mask_real_test.parquet",  index=False)

for nome in ["train", "val", "test"]:
    p = OUT_MASK_DIR / f"mask_real_{nome}.parquet"
    print(f"Salvo: {p} ({p.stat().st_size} bytes)")
print(f"Salvo: {OUT_MASK_UTILS.resolve()}")

Salvo: ..\..\Data\GoldData\Masked\mask_real_train.parquet (8527 bytes)
Salvo: ..\..\Data\GoldData\Masked\mask_real_val.parquet (8223 bytes)
Salvo: ..\..\Data\GoldData\Masked\mask_real_test.parquet (8372 bytes)
Salvo: C:\Users\jhter\OneDrive\Documents\QualiAgua\Code\3 - Preprocessing\mask_utils.py


## Síntese final

### Artefatos gerados

- `Data/GoldData/Masked/mask_real_{train,val,test}.parquet` — máscaras reais binárias, mesma ordem de linhas que os splits.
- `Code/3 - Preprocessing/mask_utils.py` — módulo com `generate_artificial_mask(M, miss_rate, seed)`.

### Protocolo confirmado

- **Treino**: a cada batch, `B = generate_artificial_mask(M_batch, MISS_RATES_PADRAO, seed=None)` — `B` muda em cada chamada, dá robustez ao gerador.
- **Validação / Test**: chamar com `seed` fixa (ex.: `seed=42`) para que GAIN, baselines (Etapa 3) e avaliação (Etapa 5) comparem **exatamente o mesmo** mascaramento.
- A **loss de reconstrução** do gerador deve usar apenas células `(M=1, B=0)` (artificialmente mascaradas) — essa é a verdade observável a recuperar.

### Recomendação propagada para `04_GAIN/`

**JC341 (n=10)** e **MR363 (n=8)** são estações descontinuadas após 2015 com posição instável no PCA (ver `resumo.md` §5). Para o treino:

- **Opção A**: peso reduzido no loss para amostras dessas estações (ex.: `weight = n_estacao / max(n_estacoes)`).
- **Opção B**: consolidá-las com a vizinha geográfica antes do treino (`JC341 → JC342`, `MR363 → MR361`).

Decisão final em `04_GAIN/01_arquitetura.ipynb`. Registrar no `split_info.json` da Etapa 5 caso a Opção B seja escolhida (gera novo split).

### Avisos para a Etapa 5 (Evaluation)

- Variáveis com **0 cobertura** em val/test (Cianobactérias em val/test, Microcistinas e Coliformes em test) só podem ser avaliadas via máscara artificial sobre o treino. Isso já foi registrado em `split_info.json["gaps_conscientes"]`.
- Comparações GAIN × baselines (KNN, MICE) **devem** usar a mesma `seed` na geração de B para que as métricas comparem o mesmo conjunto de células a reconstruir.

### Próximo passo

Fim da Etapa 2. Próximo: `Code/4 - Baselines/01_baselines_simples.ipynb` — KNN e MICE como referência para a GAIN superar.